# WeatherAPI.com NFL Weather Ingestion

Pull current, forecast, and historical weather data for NFL game locations from WeatherAPI.com.

**Features:**
- Free tier: 1 million API calls/month
- Current weather, 3-day forecast, and historical data
- Detailed wind metrics (speed, gusts, direction)
- Temperature, precipitation, visibility
- Stadium/city-level weather data

**Weather Metrics for Fantasy Impact:**
- **Wind Speed** → QB downgrade, deep ball suppression, kicker accuracy
- **Wind Gusts** → Extreme volatility for passing game
- **Wind Direction** → Crosswinds affect kickers most
- **Temperature** → Cold weather (<32°F) → RB boost, passing downgrade
- **Precipitation** → Ball handling issues, run-heavy game scripts

**Resources:**
- Website: https://www.weatherapi.com
- Docs: https://www.weatherapi.com/docs/
- Free tier: 1M calls/month
- Sign up: https://www.weatherapi.com/signup.aspx

In [0]:
import requests
import pandas as pd
import json
from datetime import datetime, timedelta
from pyspark.sql import Row
from pyspark.sql import functions as F
from pyspark.sql.types import *

# Configuration
BASE_URL = "https://api.weatherapi.com/v1"
SEASON = 2024
WEEK = 18

# Get API key from Databricks Secrets
# Create secret scope first: databricks secrets create-scope --scope api-keys
# Add key: databricks secrets put --scope api-keys --key weatherapi-key
try:
    API_KEY = dbutils.secrets.get(scope="api-keys", key="weatherapi-key")
    print("✓ API key loaded from secrets")
except:
    print("⚠️ ERROR: API key not found in secrets")
    print("\nTo set up:")
    print("1. Sign up at https://www.weatherapi.com/signup.aspx")
    print("2. Get your API key from dashboard")
    print("3. Add to secrets: databricks secrets put --scope api-keys --key weatherapi-key")
    print("4. Or set manually: API_KEY = 'your-key-here'")
    # Uncomment to set manually for testing
    # API_KEY = "your-weatherapi-key-here"
    raise

print(f"\n🌤️ WeatherAPI.com NFL Weather Ingestion")
print(f"Season: {SEASON}, Week: {WEEK}")
print(f"API Endpoint: {BASE_URL}")
print(f"Free Tier: 1M calls/month")

In [0]:
# NFL stadium locations (city names for weather lookup)
# WeatherAPI.com accepts city names, zip codes, or lat/long

NFL_STADIUMS = {
    # Team: (City, Stadium Name, Is Dome/Retractable)
    'ARI': ('Glendale, AZ', 'State Farm Stadium', True),  # Retractable roof
    'ATL': ('Atlanta, GA', 'Mercedes-Benz Stadium', True),  # Retractable roof
    'BAL': ('Baltimore, MD', 'M&T Bank Stadium', False),
    'BUF': ('Orchard Park, NY', 'Highmark Stadium', False),
    'CAR': ('Charlotte, NC', 'Bank of America Stadium', False),
    'CHI': ('Chicago, IL', 'Soldier Field', False),
    'CIN': ('Cincinnati, OH', 'Paycor Stadium', False),
    'CLE': ('Cleveland, OH', 'Cleveland Browns Stadium', False),
    'DAL': ('Arlington, TX', 'AT&T Stadium', True),  # Dome
    'DEN': ('Denver, CO', 'Empower Field at Mile High', False),
    'DET': ('Detroit, MI', 'Ford Field', True),  # Dome
    'GB': ('Green Bay, WI', 'Lambeau Field', False),
    'HOU': ('Houston, TX', 'NRG Stadium', True),  # Retractable roof
    'IND': ('Indianapolis, IN', 'Lucas Oil Stadium', True),  # Retractable roof
    'JAX': ('Jacksonville, FL', 'TIAA Bank Field', False),
    'KC': ('Kansas City, MO', 'GEHA Field at Arrowhead Stadium', False),
    'LAC': ('Inglewood, CA', 'SoFi Stadium', False),  # Has roof but open-air design
    'LAR': ('Inglewood, CA', 'SoFi Stadium', False),
    'LV': ('Las Vegas, NV', 'Allegiant Stadium', True),  # Dome
    'MIA': ('Miami Gardens, FL', 'Hard Rock Stadium', False),
    'MIN': ('Minneapolis, MN', 'U.S. Bank Stadium', True),  # Dome
    'NE': ('Foxborough, MA', 'Gillette Stadium', False),
    'NO': ('New Orleans, LA', 'Caesars Superdome', True),  # Dome
    'NYG': ('East Rutherford, NJ', 'MetLife Stadium', False),
    'NYJ': ('East Rutherford, NJ', 'MetLife Stadium', False),
    'PHI': ('Philadelphia, PA', 'Lincoln Financial Field', False),
    'PIT': ('Pittsburgh, PA', 'Acrisure Stadium', False),
    'SEA': ('Seattle, WA', 'Lumen Field', False),
    'SF': ('Santa Clara, CA', "Levi's Stadium", False),
    'TB': ('Tampa, FL', 'Raymond James Stadium', False),
    'TEN': ('Nashville, TN', 'Nissan Stadium', False),
    'WAS': ('Landover, MD', 'FedExField', False)
}

print(f"Loaded {len(NFL_STADIUMS)} NFL stadium locations")
print(f"\nDomed/Retractable stadiums (weather not impactful):")
for team, (city, stadium, is_dome) in NFL_STADIUMS.items():
    if is_dome:
        print(f"  {team}: {stadium}")

print(f"\nOutdoor stadiums: {sum(1 for _, _, is_dome in NFL_STADIUMS.values() if not is_dome)}")

In [0]:
# Fetch current weather for all NFL stadiums
print("Fetching current weather for NFL stadiums...\n")

weather_data = []

for team, (city, stadium, is_dome) in NFL_STADIUMS.items():
    try:
        # Fetch current weather
        url = f"{BASE_URL}/current.json"
        params = {
            'key': API_KEY,
            'q': city,
            'aqi': 'no'  # Don't need air quality data
        }
        
        response = requests.get(url, params=params, timeout=10)
        response.raise_for_status()
        
        data = response.json()
        
        # Extract relevant weather fields
        current = data.get('current', {})
        location = data.get('location', {})
        
        weather_record = {
            'team': team,
            'city': city,
            'stadium': stadium,
            'is_dome': is_dome,
            'timestamp': current.get('last_updated'),
            'temp_f': current.get('temp_f'),
            'temp_c': current.get('temp_c'),
            'feels_like_f': current.get('feelslike_f'),
            'wind_mph': current.get('wind_mph'),
            'wind_kph': current.get('wind_kph'),
            'wind_degree': current.get('wind_degree'),
            'wind_dir': current.get('wind_dir'),
            'gust_mph': current.get('gust_mph'),
            'gust_kph': current.get('gust_kph'),
            'pressure_mb': current.get('pressure_mb'),
            'precip_in': current.get('precip_in'),
            'humidity': current.get('humidity'),
            'cloud': current.get('cloud'),
            'visibility_miles': current.get('vis_miles'),
            'condition_text': current.get('condition', {}).get('text'),
            'condition_code': current.get('condition', {}).get('code')
        }
        
        weather_data.append(weather_record)
        print(f"✓ {team}: {temp_f}°F, Wind {wind_mph}mph {wind_dir}")
        
    except Exception as e:
        print(f"✗ {team}: Error - {e}")

# Convert to DataFrame
if weather_data:
    weather_df = pd.DataFrame(weather_data)
    print(f"\n✓ Fetched weather for {len(weather_df)} stadiums")
    print(f"\nSample data:")
    display(weather_df[['team', 'city', 'temp_f', 'wind_mph', 'gust_mph', 'condition_text']].head(10))
else:
    print("\n⚠️ No weather data fetched")
    weather_df = pd.DataFrame()

In [0]:
# Fetch forecast for upcoming game days
# WeatherAPI.com provides 3-day forecast on free tier

print("\nFetching 3-day forecast for stadiums...\n")

forecast_data = []

for team, (city, stadium, is_dome) in list(NFL_STADIUMS.items())[:5]:  # Sample 5 teams for demo
    try:
        url = f"{BASE_URL}/forecast.json"
        params = {
            'key': API_KEY,
            'q': city,
            'days': 3,  # 3-day forecast (free tier limit)
            'aqi': 'no'
        }
        
        response = requests.get(url, params=params, timeout=10)
        response.raise_for_status()
        
        data = response.json()
        
        # Extract forecast days
        forecast_days = data.get('forecast', {}).get('forecastday', [])
        
        for day in forecast_days:
            date = day.get('date')
            day_data = day.get('day', {})
            
            forecast_record = {
                'team': team,
                'city': city,
                'stadium': stadium,
                'is_dome': is_dome,
                'forecast_date': date,
                'max_temp_f': day_data.get('maxtemp_f'),
                'min_temp_f': day_data.get('mintemp_f'),
                'avg_temp_f': day_data.get('avgtemp_f'),
                'max_wind_mph': day_data.get('maxwind_mph'),
                'total_precip_in': day_data.get('totalprecip_in'),
                'avg_humidity': day_data.get('avghumidity'),
                'condition_text': day_data.get('condition', {}).get('text'),
                'chance_of_rain': day_data.get('daily_chance_of_rain'),
                'chance_of_snow': day_data.get('daily_chance_of_snow')
            }
            
            forecast_data.append(forecast_record)
        
        print(f"✓ {team}: {len(forecast_days)} day forecast")
        
    except Exception as e:
        print(f"✗ {team}: Error - {e}")

if forecast_data:
    forecast_df = pd.DataFrame(forecast_data)
    print(f"\n✓ Fetched {len(forecast_df)} forecast records")
    display(forecast_df.head(10))
else:
    print("\n⚠️ No forecast data fetched")
    forecast_df = pd.DataFrame()

In [0]:
# Calculate fantasy impact scores based on weather conditions

if 'weather_df' in locals() and len(weather_df) > 0:
    print("Calculating fantasy weather impact scores...\n")
    
    df = weather_df.copy()
    
    # Wind impact (affects passing game and kickers)
    # High wind (>15mph) = significant impact
    # Extreme wind (>20mph) = severe impact
    df['wind_impact_score'] = df.apply(lambda row: 
        0 if row['is_dome'] else
        1 if row['wind_mph'] < 10 else
        2 if row['wind_mph'] < 15 else
        3 if row['wind_mph'] < 20 else
        4,  # Severe
        axis=1
    )
    
    # Gust impact (sudden gusts worse than sustained wind for kickers)
    df['gust_impact_score'] = df.apply(lambda row:
        0 if row['is_dome'] or pd.isna(row['gust_mph']) else
        1 if row['gust_mph'] < 15 else
        2 if row['gust_mph'] < 20 else
        3 if row['gust_mph'] < 25 else
        4,  # Severe
        axis=1
    )
    
    # Cold weather impact (boosts RBs, hurts passing)
    # Extreme cold (<20°F) = significant run-heavy
    df['cold_impact_score'] = df.apply(lambda row:
        0 if row['is_dome'] else
        1 if row['temp_f'] >= 40 else
        2 if row['temp_f'] >= 32 else
        3 if row['temp_f'] >= 20 else
        4,  # Extreme cold
        axis=1
    )
    
    # Precipitation impact (fumbles, ball handling)
    df['precip_impact_score'] = df.apply(lambda row:
        0 if row['is_dome'] else
        1 if row['precip_in'] < 0.1 else
        2 if row['precip_in'] < 0.3 else
        3,  # Heavy rain/snow
        axis=1
    )
    
    # Overall weather impact (composite score)
    df['overall_weather_impact'] = (
        df['wind_impact_score'] + 
        df['gust_impact_score'] + 
        df['cold_impact_score'] + 
        df['precip_impact_score']
    )
    
    # Position-specific adjustments
    df['qb_adjustment'] = df['wind_impact_score'] * -0.05  # -5% per wind level
    df['rb_adjustment'] = df['cold_impact_score'] * 0.03   # +3% per cold level
    df['wr_adjustment'] = df['wind_impact_score'] * -0.08  # -8% per wind level (deep threats)
    df['k_adjustment'] = (df['wind_impact_score'] + df['gust_impact_score']) * -0.10  # -10% per level
    
    print("✓ Calculated impact scores\n")
    print("High-impact weather locations:")
    high_impact = df[df['overall_weather_impact'] >= 6].sort_values('overall_weather_impact', ascending=False)
    display(high_impact[['team', 'city', 'temp_f', 'wind_mph', 'gust_mph', 'overall_weather_impact', 'qb_adjustment', 'k_adjustment']])
    
    weather_impact_df = df
    
else:
    print("⚠️ No weather data to score")

In [0]:
# Transform to Spark DataFrame with fantasy-relevant schema

if 'weather_impact_df' in locals() and len(weather_impact_df) > 0:
    print("Transforming to Spark DataFrame...\n")
    
    rows = []
    for idx, row in weather_impact_df.iterrows():
        # Convert to Spark Row
        spark_row = Row(
            team=row['team'],
            city=row['city'],
            stadium=row['stadium'],
            is_dome=bool(row['is_dome']),
            timestamp=row['timestamp'],
            season=SEASON,
            week=WEEK,
            # Temperature
            temp_f=float(row['temp_f']) if pd.notna(row['temp_f']) else None,
            feels_like_f=float(row['feels_like_f']) if pd.notna(row['feels_like_f']) else None,
            # Wind
            wind_mph=float(row['wind_mph']) if pd.notna(row['wind_mph']) else None,
            wind_degree=int(row['wind_degree']) if pd.notna(row['wind_degree']) else None,
            wind_dir=str(row['wind_dir']) if pd.notna(row['wind_dir']) else None,
            gust_mph=float(row['gust_mph']) if pd.notna(row['gust_mph']) else None,
            # Conditions
            precip_in=float(row['precip_in']) if pd.notna(row['precip_in']) else None,
            humidity=int(row['humidity']) if pd.notna(row['humidity']) else None,
            visibility_miles=float(row['visibility_miles']) if pd.notna(row['visibility_miles']) else None,
            condition_text=str(row['condition_text']) if pd.notna(row['condition_text']) else None,
            # Impact scores
            wind_impact_score=int(row['wind_impact_score']),
            gust_impact_score=int(row['gust_impact_score']),
            cold_impact_score=int(row['cold_impact_score']),
            precip_impact_score=int(row['precip_impact_score']),
            overall_weather_impact=int(row['overall_weather_impact']),
            # Position adjustments
            qb_adjustment=float(row['qb_adjustment']),
            rb_adjustment=float(row['rb_adjustment']),
            wr_adjustment=float(row['wr_adjustment']),
            k_adjustment=float(row['k_adjustment']),
            # Full JSON for reference
            raw_data=json.dumps(row.to_dict(), default=str)
        )
        rows.append(spark_row)
    
    weather_spark_df = spark.createDataFrame(rows)
    print(f"✓ Created Spark DataFrame with {weather_spark_df.count()} records\n")
    display(weather_spark_df.limit(10))
    
else:
    print("⚠️ No data to transform")

In [0]:
# Write to bronze_nfl_weather table

if 'weather_spark_df' in locals():
    print("Writing to bronze_nfl_weather...\n")
    
    bronze_weather = weather_spark_df.withColumn("ingested_at", F.current_timestamp())
    bronze_weather = bronze_weather.withColumn("source", F.lit("weatherapi_com"))
    
    # Create temp view for merge
    bronze_weather.createOrReplaceTempView("weatherapi_bronze_updates")
    
    # Create table if not exists
    spark.sql("""
        CREATE TABLE IF NOT EXISTS main.fantasai.bronze_nfl_weather (
            team STRING,
            city STRING,
            stadium STRING,
            is_dome BOOLEAN,
            timestamp STRING,
            season INT,
            week INT,
            temp_f DOUBLE,
            feels_like_f DOUBLE,
            wind_mph DOUBLE,
            wind_degree INT,
            wind_dir STRING,
            gust_mph DOUBLE,
            precip_in DOUBLE,
            humidity INT,
            visibility_miles DOUBLE,
            condition_text STRING,
            wind_impact_score INT,
            gust_impact_score INT,
            cold_impact_score INT,
            precip_impact_score INT,
            overall_weather_impact INT,
            qb_adjustment DOUBLE,
            rb_adjustment DOUBLE,
            wr_adjustment DOUBLE,
            k_adjustment DOUBLE,
            raw_data STRING,
            source STRING,
            ingested_at TIMESTAMP
        )
        USING DELTA
    """)
    
    # Merge data
    spark.sql("""
        MERGE INTO main.fantasai.bronze_nfl_weather AS target
        USING weatherapi_bronze_updates AS source
        ON target.team = source.team 
            AND target.season = source.season 
            AND target.week = source.week
            AND target.timestamp = source.timestamp
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)
    
    print(f"✓ Merged {bronze_weather.count()} weather records into bronze_nfl_weather\n")
    
else:
    print("⚠️ No data to write")

In [0]:
%sql
-- Check weather data with high impact
SELECT 
    team,
    city,
    temp_f,
    wind_mph,
    gust_mph,
    wind_dir,
    condition_text,
    overall_weather_impact,
    qb_adjustment,
    rb_adjustment,
    wr_adjustment,
    k_adjustment,
    is_dome
FROM main.fantasai.bronze_nfl_weather
WHERE season = 2024 AND week = 18
ORDER BY overall_weather_impact DESC, wind_mph DESC
LIMIT 20

## WeatherAPI.com Features

### Available Endpoints (Free Tier)
1. **Current Weather** - `/current.json` - Real-time weather
2. **Forecast** - `/forecast.json` - 3-day forecast (free tier)
3. **Historical** - `/history.json` - Past weather data
4. **Sports** - `/sports.json` - Sports-specific data

### Pricing
- **Free**: 1 million calls/month
- **Pro**: $4/month - 5 million calls/month + 14-day forecast
- **Ultra**: Custom pricing - Historical data + more

### Key Weather Metrics for Fantasy Football

#### Wind Impact
- **0-10 mph**: Minimal impact
- **10-15 mph**: Slight passing game downgrade
- **15-20 mph**: Moderate impact - avoid kickers, downgrade deep threats
- **20+ mph**: Severe impact - major QB/K/WR downgrade, RB boost

#### Wind Gusts
- **Sudden gusts >20mph**: Kicker accuracy plummets
- **Crosswinds**: Worse than headwinds for kickers
- **Check wind direction vs stadium orientation**

#### Cold Weather
- **<32°F (freezing)**: Ball handling issues, RB boost (+5-10%)
- **<20°F (extreme cold)**: Major passing downgrade, heavy run game
- **Wind chill**: Use feels_like_f for player comfort

#### Precipitation
- **Rain**: Fumbles increase, passing accuracy down
- **Snow**: Run-heavy game scripts, low scoring
- **Dome games**: Ignore all weather

### Fantasy Adjustments by Position

**Quarterbacks**
- Wind >15mph: -10 to -20% projection
- Cold <32°F: -5 to -10% projection
- Heavy rain: -10% projection

**Running Backs**
- Cold <32°F: +5 to +10% projection (increased usage)
- Heavy rain: +5% projection
- Wind: Minimal impact

**Wide Receivers (Deep Threats)**
- Wind >15mph: -15 to -25% projection
- Cold <32°F: -5% projection
- Target slot receivers instead

**Kickers**
- Wind >12mph: Downgrade significantly
- Wind >20mph: Avoid if possible
- Gusts >20mph: Major risk
- Dome kickers: Always preferable

### Historical Weather Similarity
Use historical endpoint to find similar weather games:
```python
# Find games with similar weather conditions
# Match: wind speed, temp, precip within ranges
# Use for player performance pattern matching
```

### Best Practices
1. **Check weather day-before games** (Saturday for Sunday games)
2. **Monitor forecast updates** (weather changes quickly)
3. **Prioritize dome games in bad weather weeks**
4. **Stack RBs in cold/windy games**
5. **Avoid kickers and deep WRs in 15+ mph wind**